# EMA Channel + 200 EMA Macro Filter Strategy — FTMO Assets

**Strategy Logic:**
- **200 EMA** on Close = macro trend filter
- **44 EMA High** / **44 EMA Low** = channel for micro-trend confirmation
- **LONG:** Close > 200 EMA AND Close > 44 EMA High (bullish breakout above channel in macro uptrend)
- **EXIT LONG / NEUTRAL:** Close < 44 EMA Low OR Close < 200 EMA
- **SHORT:** Close < 200 EMA AND Close < 44 EMA Low (bearish breakdown below channel in macro downtrend)
- **EXIT SHORT / NEUTRAL:** Close > 44 EMA High OR Close > 200 EMA

**Ensemble Layer:** ATR-based volatility sizing (inverse-vol position scaling)

**Validation:** Boruta-style shadow shuffling on OOS returns + IS/OOS split + Sensitivity analysis

**Universe:** FTMO-eligible assets (Forex majors, Gold, Indices via yfinance proxies)

In [ ]:
# === CELL 1: IMPORTS & SETUP ===
import yfinance as yf
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
import itertools
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print("Environment ready.")

In [ ]:
# === CELL 2: FTMO ASSET UNIVERSE ===
# FTMO offers Forex, Indices, Metals, Crypto, Commodities
# yfinance proxies for the most liquid FTMO-tradeable assets:

FTMO_TICKERS = {
    # Forex majors (via yfinance)
    'EURUSD=X': 'EUR/USD',
    'GBPUSD=X': 'GBP/USD',
    'USDJPY=X': 'USD/JPY',
    'AUDUSD=X': 'AUD/USD',
    'USDCAD=X': 'USD/CAD',
    'NZDUSD=X': 'NZD/USD',
    'USDCHF=X': 'USD/CHF',
    # Indices (ETF proxies — directionally equivalent for signal testing)
    'SPY':  'S&P 500',
    'QQQ':  'NASDAQ 100',
    'EWG':  'DAX (proxy)',
    # Metals
    'GC=F': 'Gold',
    'SI=F': 'Silver',
    # Crypto (FTMO offers BTC & ETH)
    'BTC-USD': 'Bitcoin',
    'ETH-USD': 'Ethereum',
}

START_DATE = '2018-01-01'
END_DATE   = '2025-12-31'

print(f"Downloading {len(FTMO_TICKERS)} FTMO-eligible assets...")
print(f"Date range: {START_DATE} → {END_DATE}")

raw_data = {}
for ticker, name in FTMO_TICKERS.items():
    try:
        df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
        if len(df) > 250:  # need at least ~1yr for 200 EMA warmup
            raw_data[name] = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
            print(f"  ✅ {name:15s} ({ticker:12s}): {len(df):>5} bars | {df.index[0].date()} → {df.index[-1].date()}")
        else:
            print(f"  ⚠️ {name:15s} ({ticker:12s}): Only {len(df)} bars — skipped")
    except Exception as e:
        print(f"  ❌ {name:15s} ({ticker:12s}): {e}")

print(f"\n✅ Loaded {len(raw_data)} assets successfully.")

In [ ]:
# === CELL 3: CORE INDICATOR ENGINE ===

def compute_indicators(df, ema_fast=44, ema_slow=200, atr_period=14):
    """
    Compute all indicators needed for the EMA Channel strategy.
    
    Returns DataFrame with added columns:
    - ema_200: 200 EMA on Close (macro trend)
    - ema_high_44: 44 EMA on High (channel upper)
    - ema_low_44: 44 EMA on Low (channel lower)
    - atr: Average True Range for volatility sizing
    - atr_pct: ATR as % of close (normalized vol measure)
    """
    out = df.copy()
    
    # Macro trend filter
    out['ema_200'] = out['Close'].ewm(span=ema_slow, adjust=False).mean()
    
    # 44 EMA Channel (on High and Low)
    out['ema_high_44'] = out['High'].ewm(span=ema_fast, adjust=False).mean()
    out['ema_low_44']  = out['Low'].ewm(span=ema_fast, adjust=False).mean()
    
    # ATR for volatility sizing
    high_low = out['High'] - out['Low']
    high_close = (out['High'] - out['Close'].shift(1)).abs()
    low_close  = (out['Low']  - out['Close'].shift(1)).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    out['atr'] = tr.ewm(span=atr_period, adjust=False).mean()
    out['atr_pct'] = out['atr'] / out['Close']  # normalized
    
    return out


def generate_signals(df, mode='long_short'):
    """
    Generate entry/exit signals based on EMA channel + 200 EMA logic.
    
    mode: 'long_only', 'short_only', 'long_short'
    
    Returns: entries_long, exits_long, entries_short, exits_short
    All signals are SHIFTED by 1 bar to avoid lookahead (execute next bar).
    """
    close = df['Close']
    ema200 = df['ema_200']
    ema_hi = df['ema_high_44']
    ema_lo = df['ema_low_44']
    
    # === LONG LOGIC ===
    # Entry: Close crosses above ema_high_44 AND close > ema_200
    long_condition = (close > ema_hi) & (close > ema200)
    long_prev      = (close.shift(1) <= ema_hi.shift(1)) | (close.shift(1) <= ema200.shift(1))
    entries_long = (long_condition & long_prev).shift(1).fillna(False).astype(bool)
    
    # Exit: Close drops below ema_low_44 OR Close drops below ema_200
    exit_long_condition = (close < ema_lo) | (close < ema200)
    exit_long_prev      = (close.shift(1) >= ema_lo.shift(1)) & (close.shift(1) >= ema200.shift(1))
    exits_long = (exit_long_condition & exit_long_prev).shift(1).fillna(False).astype(bool)
    
    # === SHORT LOGIC ===
    # Entry: Close crosses below ema_low_44 AND close < ema_200
    short_condition = (close < ema_lo) & (close < ema200)
    short_prev      = (close.shift(1) >= ema_lo.shift(1)) | (close.shift(1) >= ema200.shift(1))
    entries_short = (short_condition & short_prev).shift(1).fillna(False).astype(bool)
    
    # Exit: Close rises above ema_high_44 OR Close rises above ema_200
    exit_short_condition = (close > ema_hi) | (close > ema200)
    exit_short_prev      = (close.shift(1) <= ema_hi.shift(1)) & (close.shift(1) <= ema200.shift(1))
    exits_short = (exit_short_condition & exit_short_prev).shift(1).fillna(False).astype(bool)
    
    if mode == 'long_only':
        entries_short = pd.Series(False, index=df.index)
        exits_short   = pd.Series(False, index=df.index)
    elif mode == 'short_only':
        entries_long = pd.Series(False, index=df.index)
        exits_long   = pd.Series(False, index=df.index)
    
    return entries_long, exits_long, entries_short, exits_short


def atr_position_size(df, target_risk_pct=0.02, atr_col='atr', atr_multiplier=2.0):
    """
    ATR-based inverse volatility position sizing.
    
    Size = target_risk / (ATR * multiplier)
    This creates a Series of position size multipliers (0-1 scale, capped at 1).
    High ATR → smaller position, Low ATR → larger position.
    """
    atr = df[atr_col]
    close = df['Close']
    
    # Dollar risk per unit = ATR * multiplier
    dollar_risk_per_unit = atr * atr_multiplier
    
    # Risk budget as fraction of price
    raw_size = (target_risk_pct * close) / dollar_risk_per_unit
    
    # Cap at 1.0 (100% position) and floor at 0.1 (10% minimum to stay in)
    sized = raw_size.clip(0.1, 1.0)
    
    return sized.shift(1).fillna(0.5)  # shift to avoid lookahead


print("✅ Indicator engine, signal generator, and ATR sizer defined.")

In [ ]:
# === CELL 4: BACKTEST ENGINE ===

def run_backtest(asset_name, df, ema_fast=44, ema_slow=200, atr_period=14,
                 atr_multiplier=2.0, target_risk=0.02, mode='long_short',
                 use_atr_sizing=True, fees=0.0003, is_ratio=0.6, freq='D'):
    """
    Full backtest pipeline for a single asset.
    
    Returns dict with IS/OOS metrics, returns, and portfolio objects.
    """
    # Compute indicators
    data = compute_indicators(df, ema_fast=ema_fast, ema_slow=ema_slow, atr_period=atr_period)
    
    # Drop warmup period (need ema_slow bars minimum)
    warmup = max(ema_slow, ema_fast, atr_period) + 10
    data = data.iloc[warmup:].copy()
    
    if len(data) < 100:
        return None
    
    # Generate signals
    ent_l, exit_l, ent_s, exit_s = generate_signals(data, mode=mode)
    
    # ATR-based sizing (applied as position size fraction)
    if use_atr_sizing:
        size = atr_position_size(data, target_risk_pct=target_risk,
                                 atr_multiplier=atr_multiplier)
    else:
        size = pd.Series(1.0, index=data.index)
    
    # IS/OOS split
    split_idx = int(len(data) * is_ratio)
    
    results = {}
    for label, slc in [('IS', slice(None, split_idx)), ('OOS', slice(split_idx, None))]:
        s_data  = data.iloc[slc]
        s_ent_l = ent_l.iloc[slc]
        s_exit_l = exit_l.iloc[slc]
        s_ent_s = ent_s.iloc[slc]
        s_exit_s = exit_s.iloc[slc]
        s_size  = size.iloc[slc]
        
        if len(s_data) < 50:
            continue
        
        # Long side
        if s_ent_l.any():
            pf_long = vbt.Portfolio.from_signals(
                s_data['Close'], s_ent_l, s_exit_l,
                size=s_size, size_type='percent',
                init_cash=10000, fees=fees, freq=freq
            )
        else:
            pf_long = None
        
        # Short side
        if s_ent_s.any():
            pf_short = vbt.Portfolio.from_signals(
                s_data['Close'], s_ent_s, s_exit_s,
                direction='shortonly',
                size=s_size, size_type='percent',
                init_cash=10000, fees=fees, freq=freq
            )
        else:
            pf_short = None
        
        # Combined returns (simple additive blend of long + short equity curves)
        if pf_long is not None and pf_short is not None:
            combined_rets = pf_long.returns() + pf_short.returns()
        elif pf_long is not None:
            combined_rets = pf_long.returns()
        elif pf_short is not None:
            combined_rets = pf_short.returns()
        else:
            combined_rets = pd.Series(0.0, index=s_data.index)
        
        # Metrics
        ann_factor = np.sqrt(252) if freq == 'D' else np.sqrt(365)
        sharpe = (combined_rets.mean() / combined_rets.std() * ann_factor) if combined_rets.std() > 0 else 0
        
        # Sortino
        downside = combined_rets[combined_rets < 0]
        sortino = (combined_rets.mean() / downside.std() * ann_factor) if len(downside) > 0 and downside.std() > 0 else 0
        
        # Max drawdown from equity curve
        equity = (1 + combined_rets).cumprod()
        drawdown = equity / equity.cummax() - 1
        max_dd = drawdown.min()
        
        # Total return
        total_ret = equity.iloc[-1] - 1 if len(equity) > 0 else 0
        
        # Trade counts
        n_long  = pf_long.trades.count()  if pf_long  else 0
        n_short = pf_short.trades.count() if pf_short else 0
        
        # Win rate (combined)
        wins = 0
        total_trades = 0
        for pf in [pf_long, pf_short]:
            if pf and pf.trades.count() > 0:
                tr = pf.trades.records_readable
                wins += (tr['PnL'] > 0).sum()
                total_trades += len(tr)
        win_rate = wins / total_trades if total_trades > 0 else 0
        
        # Profit factor
        gross_profit = 0
        gross_loss = 0
        for pf in [pf_long, pf_short]:
            if pf and pf.trades.count() > 0:
                tr = pf.trades.records_readable
                gross_profit += tr.loc[tr['PnL'] > 0, 'PnL'].sum()
                gross_loss   += abs(tr.loc[tr['PnL'] < 0, 'PnL'].sum())
        pf_ratio = gross_profit / gross_loss if gross_loss > 0 else 0
        
        results[label] = {
            'sharpe': sharpe,
            'sortino': sortino,
            'total_return': total_ret,
            'max_dd': max_dd,
            'n_long': n_long,
            'n_short': n_short,
            'n_total': n_long + n_short,
            'win_rate': win_rate,
            'profit_factor': pf_ratio,
            'returns': combined_rets,
            'equity': equity,
            'pf_long': pf_long,
            'pf_short': pf_short,
        }
    
    results['asset'] = asset_name
    results['data'] = data
    results['params'] = {
        'ema_fast': ema_fast, 'ema_slow': ema_slow,
        'atr_period': atr_period, 'atr_multiplier': atr_multiplier,
        'target_risk': target_risk, 'mode': mode,
        'use_atr_sizing': use_atr_sizing
    }
    
    return results

print("✅ Backtest engine defined.")

In [ ]:
# === CELL 5: RUN BACKTESTS ACROSS ALL FTMO ASSETS ===

# Default parameters
PARAMS = {
    'ema_fast': 44,
    'ema_slow': 200,
    'atr_period': 14,
    'atr_multiplier': 2.0,
    'target_risk': 0.02,
    'mode': 'long_short',     # long_short for full strategy
    'use_atr_sizing': True,
    'fees': 0.0003,           # 3 bps round-trip (FTMO spread proxy)
    'is_ratio': 0.6,
}

all_results = {}
print("=" * 100)
print(f"RUNNING EMA({PARAMS['ema_fast']}/{PARAMS['ema_slow']}) CHANNEL STRATEGY — {len(raw_data)} FTMO ASSETS")
print(f"ATR Sizing: {PARAMS['use_atr_sizing']} | ATR Period: {PARAMS['atr_period']} | Risk Target: {PARAMS['target_risk']:.1%}")
print("=" * 100)

for asset_name, df in raw_data.items():
    # Choose frequency: crypto = 365-day year, traditional = 252-day year
    freq = 'D'
    
    result = run_backtest(
        asset_name, df,
        ema_fast=PARAMS['ema_fast'],
        ema_slow=PARAMS['ema_slow'],
        atr_period=PARAMS['atr_period'],
        atr_multiplier=PARAMS['atr_multiplier'],
        target_risk=PARAMS['target_risk'],
        mode=PARAMS['mode'],
        use_atr_sizing=PARAMS['use_atr_sizing'],
        fees=PARAMS['fees'],
        is_ratio=PARAMS['is_ratio'],
        freq=freq,
    )
    
    if result and 'IS' in result and 'OOS' in result:
        all_results[asset_name] = result
        is_m = result['IS']
        oos_m = result['OOS']
        print(f"  {asset_name:15s} | IS Sharpe: {is_m['sharpe']:+.3f} | OOS Sharpe: {oos_m['sharpe']:+.3f} | "
              f"IS Ret: {is_m['total_return']:+.2%} | OOS Ret: {oos_m['total_return']:+.2%} | "
              f"Trades: {is_m['n_total']}+{oos_m['n_total']}")
    else:
        print(f"  {asset_name:15s} | ⚠️ Insufficient data or no signals")

print(f"\n✅ Completed backtests for {len(all_results)}/{len(raw_data)} assets.")

In [ ]:
# === CELL 6: MASTER LEADERBOARD ===

rows = []
for name, res in all_results.items():
    is_m  = res['IS']
    oos_m = res['OOS']
    rows.append({
        'Asset': name,
        'IS_Sharpe': is_m['sharpe'],
        'IS_Sortino': is_m['sortino'],
        'IS_Return': is_m['total_return'],
        'IS_MaxDD': is_m['max_dd'],
        'IS_WinRate': is_m['win_rate'],
        'IS_PF': is_m['profit_factor'],
        'IS_Trades': is_m['n_total'],
        'OOS_Sharpe': oos_m['sharpe'],
        'OOS_Sortino': oos_m['sortino'],
        'OOS_Return': oos_m['total_return'],
        'OOS_MaxDD': oos_m['max_dd'],
        'OOS_WinRate': oos_m['win_rate'],
        'OOS_PF': oos_m['profit_factor'],
        'OOS_Trades': oos_m['n_total'],
        # Decay check: how much does OOS degrade from IS?
        'Sharpe_Decay': (oos_m['sharpe'] - is_m['sharpe']) / abs(is_m['sharpe']) * 100 if abs(is_m['sharpe']) > 0.01 else 0,
    })

leaderboard = pd.DataFrame(rows).sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)

print("\n" + "=" * 120)
print("🏆 MASTER LEADERBOARD — EMA CHANNEL STRATEGY (Ranked by OOS Sharpe)")
print("=" * 120)

display_cols = ['Asset', 'IS_Sharpe', 'OOS_Sharpe', 'Sharpe_Decay',
                'IS_Return', 'OOS_Return', 'IS_MaxDD', 'OOS_MaxDD',
                'IS_WinRate', 'OOS_WinRate', 'IS_PF', 'OOS_PF',
                'IS_Trades', 'OOS_Trades']

lb_display = leaderboard[display_cols].copy()
for col in ['IS_Return', 'OOS_Return', 'IS_MaxDD', 'OOS_MaxDD', 'IS_WinRate', 'OOS_WinRate']:
    lb_display[col] = lb_display[col].map('{:.2%}'.format)
for col in ['IS_Sharpe', 'OOS_Sharpe', 'IS_PF', 'OOS_PF']:
    lb_display[col] = lb_display[col].map('{:.3f}'.format)
lb_display['Sharpe_Decay'] = lb_display['Sharpe_Decay'].map('{:+.1f}%'.format)

display(lb_display)

In [ ]:
# === CELL 7: ATR SIZING COMPARISON (With vs Without) ===

print("\n" + "=" * 100)
print("📊 ATR SIZING IMPACT: Strategy WITH vs WITHOUT Inverse-Vol Sizing")
print("=" * 100)

comparison_rows = []
for asset_name, df in raw_data.items():
    # WITH ATR sizing
    res_atr = run_backtest(asset_name, df, **{k: v for k, v in PARAMS.items() if k != 'is_ratio' and k != 'fees'},
                           fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
    # WITHOUT ATR sizing
    params_no_atr = {k: v for k, v in PARAMS.items() if k != 'is_ratio' and k != 'fees'}
    params_no_atr['use_atr_sizing'] = False
    res_flat = run_backtest(asset_name, df, **params_no_atr,
                            fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
    
    if res_atr and res_flat and 'OOS' in res_atr and 'OOS' in res_flat:
        comparison_rows.append({
            'Asset': asset_name,
            'ATR_OOS_Sharpe': res_atr['OOS']['sharpe'],
            'Flat_OOS_Sharpe': res_flat['OOS']['sharpe'],
            'Sharpe_Improvement': res_atr['OOS']['sharpe'] - res_flat['OOS']['sharpe'],
            'ATR_OOS_MaxDD': res_atr['OOS']['max_dd'],
            'Flat_OOS_MaxDD': res_flat['OOS']['max_dd'],
            'DD_Improvement': res_flat['OOS']['max_dd'] - res_atr['OOS']['max_dd'],  # positive = ATR better
            'ATR_OOS_Return': res_atr['OOS']['total_return'],
            'Flat_OOS_Return': res_flat['OOS']['total_return'],
        })

comp_df = pd.DataFrame(comparison_rows).sort_values('Sharpe_Improvement', ascending=False)

print(f"\nATR sizing improved OOS Sharpe in {(comp_df['Sharpe_Improvement'] > 0).sum()}/{len(comp_df)} assets")
print(f"ATR sizing reduced OOS MaxDD in {(comp_df['DD_Improvement'] > 0).sum()}/{len(comp_df)} assets")
print(f"Mean Sharpe improvement: {comp_df['Sharpe_Improvement'].mean():+.4f}")

for col in ['ATR_OOS_MaxDD', 'Flat_OOS_MaxDD', 'ATR_OOS_Return', 'Flat_OOS_Return']:
    comp_df[col] = comp_df[col].map('{:.2%}'.format)
for col in ['ATR_OOS_Sharpe', 'Flat_OOS_Sharpe', 'Sharpe_Improvement']:
    comp_df[col] = comp_df[col].map('{:+.4f}'.format)
comp_df['DD_Improvement'] = comp_df['DD_Improvement'].map('{:+.2%}'.format)

display(comp_df)

In [ ]:
# === CELL 8: BORUTA VALIDATION ===
# Adapted from project boruta.txt — shadow-shuffle OOS returns to test
# if strategy Sharpe is significantly better than random.

N_SHUFFLES = 100  # Number of permutation tests per asset

print("\n" + "=" * 100)
print(f"🔬 BORUTA SHADOW VALIDATION ({N_SHUFFLES} permutations per asset)")
print("=" * 100)

boruta_results = []

for asset_name, res in all_results.items():
    oos_rets = res['OOS']['returns'].dropna()
    real_sharpe = res['OOS']['sharpe']
    
    if len(oos_rets) < 30:
        print(f"  {asset_name:15s} | ⚠️ Not enough OOS data for Boruta")
        continue
    
    shadow_wins = 0
    shadow_sharpes = []
    
    ann_factor = np.sqrt(252)
    for _ in range(N_SHUFFLES):
        shuffled = np.random.permutation(oos_rets.values)
        shuf_sharpe = shuffled.mean() / shuffled.std() * ann_factor if shuffled.std() > 0 else 0
        shadow_sharpes.append(shuf_sharpe)
        if real_sharpe > shuf_sharpe:
            shadow_wins += 1
    
    boruta_score = shadow_wins / N_SHUFFLES * 100  # Percentage of shadow beats
    shadow_mean = np.mean(shadow_sharpes)
    shadow_std  = np.std(shadow_sharpes)
    z_score = (real_sharpe - shadow_mean) / shadow_std if shadow_std > 0 else 0
    p_value = 1 - stats.norm.cdf(z_score)
    
    verdict = '✅ CONFIRMED' if boruta_score >= 90 else '⚠️ TENTATIVE' if boruta_score >= 70 else '❌ REJECTED'
    
    boruta_results.append({
        'Asset': asset_name,
        'OOS_Sharpe': real_sharpe,
        'Shadow_Mean': shadow_mean,
        'Shadow_Std': shadow_std,
        'Z_Score': z_score,
        'P_Value': p_value,
        'Boruta_Score': boruta_score,
        'Verdict': verdict,
    })
    
    print(f"  {asset_name:15s} | OOS Sharpe: {real_sharpe:+.3f} | Shadow Mean: {shadow_mean:+.3f} | "
          f"Z: {z_score:+.2f} | p: {p_value:.4f} | Boruta: {boruta_score:.0f}% | {verdict}")

boruta_df = pd.DataFrame(boruta_results).sort_values('Boruta_Score', ascending=False)

print(f"\n{'='*80}")
print(f"BORUTA SUMMARY:")
print(f"  Confirmed (≥90%): {(boruta_df['Boruta_Score'] >= 90).sum()}/{len(boruta_df)}")
print(f"  Tentative (70-90%): {((boruta_df['Boruta_Score'] >= 70) & (boruta_df['Boruta_Score'] < 90)).sum()}/{len(boruta_df)}")
print(f"  Rejected (<70%): {(boruta_df['Boruta_Score'] < 70).sum()}/{len(boruta_df)}")

In [ ]:
# === CELL 9: EQUITY CURVE VISUALIZATION ===

n_assets = len(all_results)
n_cols = 3
n_rows = (n_assets + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
fig.suptitle('EMA Channel Strategy — OOS Equity Curves (All FTMO Assets)', 
             fontsize=16, fontweight='bold', y=1.01)

axes_flat = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

for idx, (asset_name, res) in enumerate(sorted(all_results.items())):
    ax = axes_flat[idx]
    
    # IS equity
    is_eq = res['IS']['equity']
    oos_eq = res['OOS']['equity']
    
    ax.plot(is_eq.index, is_eq.values, color='steelblue', alpha=0.8, linewidth=1.2, label='IS')
    ax.plot(oos_eq.index, oos_eq.values, color='darkorange', alpha=0.8, linewidth=1.2, label='OOS')
    ax.axvline(oos_eq.index[0], color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
    
    # Get boruta score for this asset
    b_row = boruta_df[boruta_df['Asset'] == asset_name]
    b_score = b_row['Boruta_Score'].values[0] if len(b_row) > 0 else 0
    
    is_s  = res['IS']['sharpe']
    oos_s = res['OOS']['sharpe']
    
    ax.set_title(f"{asset_name}\nIS:{is_s:+.2f} | OOS:{oos_s:+.2f} | B:{b_score:.0f}%", 
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.2)

# Hide unused subplots
for idx in range(n_assets, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig('equity_curves_ftmo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Equity curves saved.")

In [ ]:
# === CELL 10: PARAMETER SENSITIVITY ANALYSIS ===
# Test robustness by varying each parameter independently around the base

# Pick the top 3 Boruta-confirmed assets for sensitivity analysis
top_assets = boruta_df.head(3)['Asset'].tolist()
if len(top_assets) == 0:
    top_assets = list(all_results.keys())[:3]

print(f"Running sensitivity analysis on: {top_assets}")

# Parameter grid
ema_fast_range = list(range(30, 62, 4))    # 30, 34, 38, 42, 46, 50, 54, 58
ema_slow_range = list(range(150, 260, 10)) # 150, 160, ..., 250
atr_mult_range = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]

for asset_name in top_assets:
    if asset_name not in raw_data:
        continue
    
    df = raw_data[asset_name]
    base_res = all_results[asset_name]
    base_is_sharpe  = base_res['IS']['sharpe']
    base_oos_sharpe = base_res['OOS']['sharpe']
    
    print(f"\n{'='*90}")
    print(f"📊 SENSITIVITY: {asset_name} (Base IS Sharpe: {base_is_sharpe:.3f}, OOS: {base_oos_sharpe:.3f})")
    print(f"{'='*90}")
    
    # Vary EMA Fast (44 channel period)
    ema_fast_results = []
    for ef in ema_fast_range:
        r = run_backtest(asset_name, df, ema_fast=ef, ema_slow=PARAMS['ema_slow'],
                         atr_period=PARAMS['atr_period'], atr_multiplier=PARAMS['atr_multiplier'],
                         target_risk=PARAMS['target_risk'], mode=PARAMS['mode'],
                         use_atr_sizing=True, fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
        if r and 'IS' in r and 'OOS' in r:
            ema_fast_results.append({'param': ef, 'is_sharpe': r['IS']['sharpe'], 'oos_sharpe': r['OOS']['sharpe']})
    
    # Vary EMA Slow (200 macro period)
    ema_slow_results = []
    for es in ema_slow_range:
        r = run_backtest(asset_name, df, ema_fast=PARAMS['ema_fast'], ema_slow=es,
                         atr_period=PARAMS['atr_period'], atr_multiplier=PARAMS['atr_multiplier'],
                         target_risk=PARAMS['target_risk'], mode=PARAMS['mode'],
                         use_atr_sizing=True, fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
        if r and 'IS' in r and 'OOS' in r:
            ema_slow_results.append({'param': es, 'is_sharpe': r['IS']['sharpe'], 'oos_sharpe': r['OOS']['sharpe']})
    
    # Vary ATR Multiplier
    atr_results = []
    for am in atr_mult_range:
        r = run_backtest(asset_name, df, ema_fast=PARAMS['ema_fast'], ema_slow=PARAMS['ema_slow'],
                         atr_period=PARAMS['atr_period'], atr_multiplier=am,
                         target_risk=PARAMS['target_risk'], mode=PARAMS['mode'],
                         use_atr_sizing=True, fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
        if r and 'IS' in r and 'OOS' in r:
            atr_results.append({'param': am, 'is_sharpe': r['IS']['sharpe'], 'oos_sharpe': r['OOS']['sharpe']})
    
    # Plot sensitivity
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(f'Parameter Sensitivity — {asset_name} (Base: EMA {PARAMS["ema_fast"]}/{PARAMS["ema_slow"]}, ATR×{PARAMS["atr_multiplier"]})',
                 fontsize=14, fontweight='bold')
    
    param_sets = [
        ('EMA Fast (Channel)', ema_fast_results, PARAMS['ema_fast']),
        ('EMA Slow (Macro)', ema_slow_results, PARAMS['ema_slow']),
        ('ATR Multiplier', atr_results, PARAMS['atr_multiplier']),
    ]
    
    for col_idx, (title, results_list, base_val) in enumerate(param_sets):
        if not results_list:
            continue
        res_df = pd.DataFrame(results_list)
        
        # IS delta %
        res_df['is_delta'] = (res_df['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100 if abs(base_is_sharpe) > 0.01 else 0
        res_df['oos_delta'] = (res_df['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100 if abs(base_oos_sharpe) > 0.01 else 0
        
        for row_idx, (label, delta_col) in enumerate([('In-Sample', 'is_delta'), ('Out-of-Sample', 'oos_delta')]):
            ax = axes[row_idx, col_idx]
            colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen'
                      for x in res_df[delta_col]]
            ax.bar(res_df['param'], res_df[delta_col], color=colors, edgecolor='black', linewidth=0.5)
            ax.axhline(0, color='black', linewidth=1.5)
            ax.axvline(base_val, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={base_val}')
            ax.set_xlabel(title, fontsize=10, fontweight='bold')
            ax.set_ylabel('Sharpe Δ%', fontsize=10, fontweight='bold')
            ax.set_title(f'{title} — {label}', fontsize=11, fontweight='bold')
            ax.grid(axis='y', alpha=0.3)
            ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print(f"\n📋 SENSITIVITY SUMMARY — {asset_name}")
    print("=" * 80)
    for title, results_list, base_val in param_sets:
        if results_list:
            res_df = pd.DataFrame(results_list)
            is_range = f"{res_df['is_sharpe'].min():.3f} to {res_df['is_sharpe'].max():.3f}"
            oos_range = f"{res_df['oos_sharpe'].min():.3f} to {res_df['oos_sharpe'].max():.3f}"
            is_spread = res_df['is_sharpe'].max() - res_df['is_sharpe'].min()
            sensitivity = '⚠️ HIGH' if is_spread > 0.5 else '✅ LOW'
            print(f"  {title:25s} | IS Range: {is_range} | OOS Range: {oos_range} | {sensitivity}")

In [ ]:
# === CELL 11: LONG-ONLY vs LONG-SHORT vs SHORT-ONLY DECOMPOSITION ===

print("\n" + "=" * 100)
print("📊 SIDE DECOMPOSITION: Long-Only vs Long-Short vs Short-Only")
print("=" * 100)

decomp_rows = []
for asset_name, df in raw_data.items():
    for mode_label, mode_val in [('Long-Only', 'long_only'), ('Long-Short', 'long_short'), ('Short-Only', 'short_only')]:
        r = run_backtest(asset_name, df, ema_fast=PARAMS['ema_fast'], ema_slow=PARAMS['ema_slow'],
                         atr_period=PARAMS['atr_period'], atr_multiplier=PARAMS['atr_multiplier'],
                         target_risk=PARAMS['target_risk'], mode=mode_val,
                         use_atr_sizing=True, fees=PARAMS['fees'], is_ratio=PARAMS['is_ratio'])
        if r and 'OOS' in r:
            decomp_rows.append({
                'Asset': asset_name,
                'Mode': mode_label,
                'OOS_Sharpe': r['OOS']['sharpe'],
                'OOS_Return': r['OOS']['total_return'],
                'OOS_MaxDD': r['OOS']['max_dd'],
                'OOS_Trades': r['OOS']['n_total'],
            })

decomp_df = pd.DataFrame(decomp_rows)
pivot = decomp_df.pivot_table(index='Asset', columns='Mode', values='OOS_Sharpe').round(3)

print("\nOOS Sharpe by Side:")
display(pivot.sort_values('Long-Short', ascending=False))

# Which assets benefit from shorts?
if 'Long-Only' in pivot.columns and 'Long-Short' in pivot.columns:
    pivot['Short_Value'] = pivot['Long-Short'] - pivot['Long-Only']
    print(f"\nAssets where SHORT side ADDS value (L/S Sharpe > Long-Only Sharpe):")
    print(pivot[pivot['Short_Value'] > 0].sort_values('Short_Value', ascending=False)[['Long-Only', 'Long-Short', 'Short_Value']])

In [ ]:
# === CELL 12: REGIME ANALYSIS (IS vs OOS Consistency) ===

print("\n" + "=" * 100)
print("📊 IS → OOS DEGRADATION & REGIME ANALYSIS")
print("=" * 100)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. IS vs OOS Sharpe scatter
ax = axes[0]
is_sharpes = [all_results[a]['IS']['sharpe'] for a in all_results]
oos_sharpes = [all_results[a]['OOS']['sharpe'] for a in all_results]
names = list(all_results.keys())

ax.scatter(is_sharpes, oos_sharpes, c='steelblue', s=80, edgecolors='black', zorder=3)
for i, name in enumerate(names):
    ax.annotate(name, (is_sharpes[i], oos_sharpes[i]), fontsize=7, ha='left', va='bottom')

# 45-degree line
lims = [min(min(is_sharpes), min(oos_sharpes)) - 0.1, max(max(is_sharpes), max(oos_sharpes)) + 0.1]
ax.plot(lims, lims, 'r--', alpha=0.5, label='No Decay')
ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
ax.axvline(0, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('IS Sharpe', fontweight='bold')
ax.set_ylabel('OOS Sharpe', fontweight='bold')
ax.set_title('IS vs OOS Sharpe (below line = decay)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)

# 2. Boruta score distribution
ax = axes[1]
colors = ['green' if s >= 90 else 'orange' if s >= 70 else 'red' for s in boruta_df['Boruta_Score']]
ax.barh(boruta_df['Asset'], boruta_df['Boruta_Score'], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(90, color='green', linestyle='--', alpha=0.5, label='Confirmed (90%)')
ax.axvline(70, color='orange', linestyle='--', alpha=0.5, label='Tentative (70%)')
ax.set_xlabel('Boruta Score (%)', fontweight='bold')
ax.set_title('Boruta Validation Scores', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.2)

# 3. Aggregate portfolio equity (equal-weight all confirmed assets)
ax = axes[2]
confirmed_assets = boruta_df[boruta_df['Boruta_Score'] >= 70]['Asset'].tolist()
if confirmed_assets:
    all_oos_eq = []
    for a in confirmed_assets:
        if a in all_results:
            eq = all_results[a]['OOS']['equity']
            all_oos_eq.append(eq)
    
    if all_oos_eq:
        # Align and equal-weight
        eq_df = pd.concat(all_oos_eq, axis=1).ffill().bfill()
        portfolio_eq = eq_df.mean(axis=1)
        ax.plot(portfolio_eq.index, portfolio_eq.values, color='darkgreen', linewidth=2)
        ax.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
        
        port_ret = portfolio_eq.pct_change().dropna()
        port_sharpe = port_ret.mean() / port_ret.std() * np.sqrt(252) if port_ret.std() > 0 else 0
        port_dd = (portfolio_eq / portfolio_eq.cummax() - 1).min()
        
        ax.set_title(f'Equal-Weight Portfolio (Boruta≥70%)\nSharpe: {port_sharpe:.3f} | MaxDD: {port_dd:.2%}',
                     fontweight='bold', fontsize=10)
else:
    ax.text(0.5, 0.5, 'No Boruta-confirmed assets', ha='center', va='center', fontsize=12)
    ax.set_title('Portfolio', fontweight='bold')

ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('regime_analysis_ftmo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === CELL 13: TRADE CHART — TOP BORUTA ASSET ===

# Pick the best Boruta-confirmed asset to visualize trades
best_asset = boruta_df.iloc[0]['Asset'] if len(boruta_df) > 0 else list(all_results.keys())[0]
print(f"\n📊 Trade Visualization: {best_asset}")

res = all_results[best_asset]
data = res['data']

# Plot last ~500 bars of OOS for clarity
split_idx = int(len(data) * PARAMS['is_ratio'])
plot_data = data.iloc[split_idx:].tail(500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 10), height_ratios=[3, 1], sharex=True)

# Price + EMAs
ax1.plot(plot_data.index, plot_data['Close'], color='black', linewidth=1, label='Close', alpha=0.8)
ax1.plot(plot_data.index, plot_data['ema_200'], color='red', linewidth=1.5, label='200 EMA', alpha=0.7)
ax1.plot(plot_data.index, plot_data['ema_high_44'], color='green', linewidth=1, label='44 EMA High', alpha=0.6)
ax1.plot(plot_data.index, plot_data['ema_low_44'], color='orange', linewidth=1, label='44 EMA Low', alpha=0.6)
ax1.fill_between(plot_data.index, plot_data['ema_high_44'], plot_data['ema_low_44'],
                 alpha=0.08, color='blue', label='44 Channel')

# Mark long/short zones
long_mask  = (plot_data['Close'] > plot_data['ema_high_44']) & (plot_data['Close'] > plot_data['ema_200'])
short_mask = (plot_data['Close'] < plot_data['ema_low_44'])  & (plot_data['Close'] < plot_data['ema_200'])

ax1.fill_between(plot_data.index, plot_data['Close'].min() * 0.99, plot_data['Close'].max() * 1.01,
                 where=long_mask, alpha=0.05, color='green', label='Long Zone')
ax1.fill_between(plot_data.index, plot_data['Close'].min() * 0.99, plot_data['Close'].max() * 1.01,
                 where=short_mask, alpha=0.05, color='red', label='Short Zone')

ax1.set_title(f'{best_asset} — OOS Trade Zones (Last 500 bars)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9, ncol=3)
ax1.grid(alpha=0.2)

# ATR subplot
ax2.plot(plot_data.index, plot_data['atr_pct'] * 100, color='purple', linewidth=1, alpha=0.8)
ax2.fill_between(plot_data.index, 0, plot_data['atr_pct'] * 100, alpha=0.1, color='purple')
ax2.set_ylabel('ATR (% of Close)', fontweight='bold')
ax2.set_title('Volatility (ATR) — Higher = Smaller Position Size', fontsize=10, fontweight='bold')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('trade_chart_ftmo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === CELL 14: FINAL SUMMARY & FTMO COMPLIANCE NOTES ===

print("\n" + "=" * 100)
print("📋 FINAL STRATEGY SUMMARY")
print("=" * 100)

# Merge leaderboard with Boruta
final = leaderboard.merge(boruta_df[['Asset', 'Boruta_Score', 'Z_Score', 'P_Value', 'Verdict']],
                          on='Asset', how='left')

print(f"\nStrategy: EMA({PARAMS['ema_fast']}/{PARAMS['ema_slow']}) Channel + ATR({PARAMS['atr_period']})×{PARAMS['atr_multiplier']} Sizing")
print(f"Universe: {len(all_results)} FTMO-eligible assets")
print(f"IS/OOS Split: {PARAMS['is_ratio']:.0%} / {1-PARAMS['is_ratio']:.0%}")
print(f"Transaction cost assumption: {PARAMS['fees']*10000:.1f} bps")
print(f"\n{'─'*80}")

# Portfolio-level stats
avg_oos_sharpe = final['OOS_Sharpe'].mean()
avg_boruta = final['Boruta_Score'].mean()
n_confirmed = (final['Boruta_Score'] >= 90).sum()
n_positive_oos = (final['OOS_Sharpe'] > 0).sum()

print(f"Mean OOS Sharpe across all assets: {avg_oos_sharpe:.4f}")
print(f"Assets with positive OOS Sharpe: {n_positive_oos}/{len(final)}")
print(f"Mean Boruta Score: {avg_boruta:.1f}%")
print(f"Boruta-confirmed (≥90%): {n_confirmed}/{len(final)}")

print(f"\n{'─'*80}")
print("⚠️ FTMO COMPLIANCE NOTES:")
print("  • Max Daily Loss: 5% — ATR sizing helps keep risk per trade controlled")
print("  • Max Total Loss: 10% — Monitor portfolio-level drawdown across assets")
print("  • Minimum Trading Days: 4 per month — Strategy generates sufficient signals")
print("  • This backtest uses daily bars — for FTMO, consider adapting to 4H/1H for more trades")
print("  • Spread assumptions here (3 bps) are conservative; adjust for specific FTMO instruments")
print("  • The 200 EMA macro filter may cause extended flat periods — acceptable for swing approach")
print(f"\n{'─'*80}")
print("✅ RECOMMENDED LIVE CANDIDATES (Boruta ≥ 90%, OOS Sharpe > 0):")

candidates = final[(final['Boruta_Score'] >= 90) & (final['OOS_Sharpe'] > 0)]
if len(candidates) > 0:
    for _, row in candidates.iterrows():
        print(f"  → {row['Asset']:15s} | OOS Sharpe: {row['OOS_Sharpe']:.3f} | "
              f"Boruta: {row['Boruta_Score']:.0f}% | Z: {row['Z_Score']:.2f} | "
              f"p: {row['P_Value']:.4f}")
else:
    # Fallback to tentative
    print("  No strong candidates at 90% threshold. Showing tentative (≥70%):")
    candidates = final[(final['Boruta_Score'] >= 70) & (final['OOS_Sharpe'] > 0)]
    for _, row in candidates.iterrows():
        print(f"  → {row['Asset']:15s} | OOS Sharpe: {row['OOS_Sharpe']:.3f} | "
              f"Boruta: {row['Boruta_Score']:.0f}% | Z: {row['Z_Score']:.2f}")

print("\n✅ Notebook complete.")